In [1]:
from langgraph.graph import StateGraph,START,END
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict

In [2]:
load_dotenv()

True

In [3]:
model=ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temperature=0
)

In [4]:
class StateLLM(TypedDict):

    question: str
    answer: str

In [5]:
def llm_qa(state : StateLLM) -> StateLLM:

    question = state['question']

    # form a prompt
    prompt = f'Answer the following question {question}'

    # ask that question to the LLM
    answer = model.invoke(prompt).content

    # update the answer in the state
    state['answer'] = answer

    return state

In [6]:
# create our graph

graph = StateGraph(StateLLM)

# add nodes
graph.add_node('llm_qa', llm_qa)

# add edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

# compile
workflow = graph.compile()

In [7]:
# execute

intial_state = {'question': 'How far is moon from the earth?'}

final_state = workflow.invoke(intial_state)

print(final_state['answer'])

[{'type': 'text', 'text': 'On average, the Moon is about **238,855 miles (384,400 kilometers)** away from Earth. \n\nBecause the Moon travels in an elliptical (oval-shaped) orbit, this distance changes over the course of a month:\n\n* **At its closest (Perigee):** About 225,623 miles (363,104 km)\n* **At its farthest (Apogee):** About 252,088 miles (405,696 km)\n\nTo put that into perspective, you could fit roughly **30 Earths** into the space between the Earth and the Moon.', 'extras': {'signature': 'EqIGCp8GAWkUfRMEI6+/cFRLs7OhTJP7rCJmNOkQxzVc8+lXwJ8Ksrkw8hcMA3uwklMHHhvT4MegTvROzm0ryauRcNxfUrv58lpZbMZctdj5p7piT3EXPmEIS0sPrO3nGk1kfv1R1avCpNxkBi5y2zjcexpW8t5lJMNQZuhaN0wNDKX2yPyZlVu+/cSfJrQus2/sWmvHgNobx/2jchBNYHjsEnWOBt7+fkFR0MJCSq2XQFYMNWmp+WsNpnfnigmwiQlL/SFiT7UMeBsSh4z+PlzwnsHcKchIgwg7Wkfy/QCHonaBkDCu6nKjaikEq5lKRqJgK/s+NGmO+8ROtkzQHLFgKKhKoEqlCbK/0TTs2kgruCHx9M6g1vplliA5hLiOx9sSyGideoWxOZwLZavXEc7V/FVNlkxD3SFHHIFsD15441wcZK27TEO+YtbigdVFR2VCuNZmN1WOtVZVQUEZEAWyokhpsYuFt9P7DfEVpux41